In [1]:
import pandas as pd 
import torch
import torch.nn as nn
import torch.optim as optim

import math
import numpy as np
import pandas as pd
import random
import re

from torch.utils.data import DataLoader, Dataset

df = pd.read_csv('final_drug_design.csv')
df.shape

(1485927, 3)

In [2]:
df = df[~df['scaffold'].isnull()]
df.shape

(1485927, 3)

In [3]:
df = df[df['scaffold'] != df['SMILES']]

In [4]:
df = df.reset_index()
df.head()

,index,SMILES,ID,scaffold
0,0,Cn1ccnc1,CHEMBL543,c1c[nH]cn1
1,1,CN1CCCC1,CHEMBL665,C1CCNC1
2,5,Nc1ccccc1,CHEMBL538,c1ccccc1
3,12,N#CN1CCC1,CHEMBL8123,C1CNC1
4,13,Cc1ccccc1,CHEMBL9113,c1ccccc1


In [5]:
df = df.drop('index',axis=1)

In [6]:
df.head()

,SMILES,ID,scaffold
0,Cn1ccnc1,CHEMBL543,c1c[nH]cn1
1,CN1CCCC1,CHEMBL665,C1CCNC1
2,Nc1ccccc1,CHEMBL538,c1ccccc1
3,N#CN1CCC1,CHEMBL8123,C1CNC1
4,Cc1ccccc1,CHEMBL9113,c1ccccc1


In [7]:
from rdkit import Chem

# 원래 SMILES (여러 가지 표현이 가능)
canonical_smiles_list = []

for s in df['SMILES']:
    can_smi = Chem.MolToSmiles(Chem.MolFromSmiles(s), canonical=True)  # Canonical SMILES 변환
    canonical_smiles_list.append(can_smi)

In [8]:
df['cano_smiles'] = canonical_smiles_list

In [9]:
cano_scaffold = []

for s in df['scaffold']:
	try:
		can_smi = Chem.MolToSmiles(Chem.MolFromSmiles(s), canonical=True)  # Canonical SMILES 변환
		cano_scaffold.append(can_smi)
	except:
		cano_scaffold.append('유효하지 않음')

[14:23:55] Can't kekulize mol.  Unkekulized atoms: 2
[14:24:11] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4
[14:24:12] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8 9 10 11 12
[14:24:20] Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 6 7 8 9 10
[14:24:28] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8 9 10 11 12
[14:24:29] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8 9 10 11 12
[14:24:30] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8
[14:24:31] Can't kekulize mol.  Unkekulized atoms: 4
[14:24:35] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8 9 10 11 12
[14:24:35] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8 9 10 11 12
[14:24:37] Can't kekulize mol.  Unkekulized atoms: 2 6 7 8 9 10 11
[14:24:37] Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 6 7 8 9 10 11 12 13 14
[14:24:37] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8 9 10 11 12
[14:24:37] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 

In [10]:
df['cano_scaffold'] = cano_scaffold

In [11]:
df = df[df['cano_scaffold'] != '유효하지 않음']

In [12]:
import numpy as np
np.array([len(i) for i in df['cano_scaffold']]).mean()

36.468207122256985

In [13]:
df = df[
    (df['cano_smiles'].str.len() >= 30) & 
    (df['cano_smiles'].str.len() <= 50)
]

In [24]:
df.to_csv('for_transformer_train.csv',index=False)
df.shape

(804914, 5)

In [14]:
import selfies as sf

scaffold_encodered = []
smiles_encodered = []

for i in range(len(df['cano_scaffold'])):
	try:
		scaffold_encode = sf.encoder(df['cano_scaffold'].iloc[i])  # smiles 인코딩
		scaffold_encode_split = list(sf.split_selfies(scaffold_encode))
		scaffold_encodered.append([df['cano_scaffold'].iloc[i], scaffold_encode_split])
  
		smiles_encode = sf.encoder(df['cano_smiles'].iloc[i])  # smiles 인코딩
		smiles_encode_split = list(sf.split_selfies(smiles_encode))
		smiles_encodered.append([df['cano_smiles'].iloc[i], smiles_encode_split])
	except:
		continue

	if len(scaffold_encodered) == 10000:
		break

In [15]:
x_encoded = [['[SOS]'] + i[1] + (38 - len(i[1]))*['[PAD]'] + ['[EOS]'] for i in scaffold_encodered]
y_encoded = [['[SOS]'] + i[1] + (38 - len(i[1]))*['[PAD]'] + ['[EOS]'] for i in smiles_encodered]

In [16]:
vocab = []

for i in x_encoded:
    vocab += i

for j in y_encoded:
    vocab += j

vocab = list(set(vocab))
token2id = {tok: i for i, tok in enumerate(vocab)}

In [17]:
x_ids = []
y_ids = []

for x_seq in x_encoded:
	x_ids.append([token2id[tok] for tok in x_seq])

for y_seq in y_encoded:
	y_ids.append([token2id[tok] for tok in y_seq])

In [20]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X_tensor = torch.tensor(x_ids, dtype=torch.long)
y_tensor = torch.tensor(y_ids, dtype=torch.long)

dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)